<a href="https://colab.research.google.com/github/Vatsal-Savani-006/Machine-Learning-Lab-Repo/blob/main/Vatsal-%20Practical5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Write a Program to Demonstrate the Working of the Decision Tree Based Id3 Algo, Use Appropriate Dataset for This.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("preetgogna/play-tennis")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'play-tennis' dataset.
Path to dataset files: /kaggle/input/play-tennis


In [ ]:
import numpy as np
import pandas as pd
df = pd.read_csv(path + '/PlayTennis.csv')
df.head()

,Outlook,Temperature,Humidity,Wind,Play Tennis
0,Sunny,Hot,High,Weak,No
1,Sunny,Hot,High,Strong,No
2,Overcast,Hot,High,Weak,Yes
3,Rain,Mild,High,Weak,Yes
4,Rain,Cool,Normal,Weak,Yes


In [ ]:
def entropy(column):
  values, counts = np.unique(column, return_counts=True)
  total_counts = len(column)
  entropy = 0
  for count in counts:
    prob = count/total_counts
    if prob>0:
      entropy -= prob * np.log2(prob)
  return entropy

In [ ]:
def information_gain(data, feature, target="Play"):
    total_entropy = entropy(data[target])

    values, counts = np.unique(data[feature], return_counts=True)

    weighted_entropy = 0
    for i in range(len(values)):
        subset = data[data[feature] == values[i]]
        weighted_entropy += (counts[i] / sum(counts)) * entropy(subset[target])

    return total_entropy - weighted_entropy
for col in df.columns[:-1]:
    print(col, ":", information_gain(df, col))

Outlook : 0.24674981977443933
Temp : 0.02922256565895487
Humidity : 0.15183550136234159
Wind : 0.04812703040826949


In [ ]:
def id3(data, original_data, features, target="Play", parent_node=None):

    # If all target values same → return that value
    if len(np.unique(data[target])) <= 1:
        return np.unique(data[target])[0]

    # If dataset empty → return majority class of original dataset
    elif len(data) == 0:
        return np.unique(original_data[target])[np.argmax(
            np.unique(original_data[target], return_counts=True)[1])]

    # If no features left → return parent node
    elif len(features) == 0:
        return parent_node

    else:
        parent_node = np.unique(data[target])[np.argmax(
            np.unique(data[target], return_counts=True)[1])]

        # Select best feature
        gains = [information_gain(data, feature, target) for feature in features]
        best_feature = features[np.argmax(gains)]

        tree = {best_feature: {}}

        features = [f for f in features if f != best_feature]

        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = id3(sub_data, original_data, features, target, parent_node)
            tree[best_feature][value] = subtree

        return tree
features = df.columns[:-1].tolist()
tree = id3(df, df, features)

print("Decision Tree:")
print(tree)

Decision Tree:
{'Outlook': {'Overcast': 'Yes', 'Rain': {'Wind': {'Strong': 'No', 'Weak': 'Yes'}}, 'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}}}


In [ ]:
def predict(query, tree):
    for key in list(query.keys()):
        if key in tree:
            value = query[key]
            subtree = tree[key][value]

            if isinstance(subtree, dict):
                return predict(query, subtree)
            else:
                return subtree
new_sample = {
    'Outlook': 'Sunny',
    'Temp': 'Cool',
    'Humidity': 'Normal',
    'Wind': 'Weak'
}

result = predict(new_sample, tree)
print("Prediction:", result)

Prediction: Yes
